# Sampling Variability of Google Trends RSV Data

This notebook assesses the **sampling variability** of Google Trends Relative Search Volume (RSV) data
by comparing three independent data pulls:

| Pull | Label | Date | Interval |
|------|-------|------|----------|
| 0 | Pull 0 | 17 Apr 2026 | — |
| 1 | Pull 1 | 05 Nov 2026, 06:59 | ~7 months after Pull 0 |
| 2 | Pull 2 | 05 Nov 2026, 09:03 | ~2 hours after Pull 1 |

**Metrics:**
- **MAD** (Mean Absolute Difference): average |RSV₁ − RSV₂| across months — primary measure of magnitude of variability
- **RMSD** (Root Mean Square Difference): √(mean((RSV₁−RSV₂)²)) — sensitive to larger deviations
- **Pearson r**: correlation between temporal patterns — complementary measure of pattern stability

**Pairwise comparisons:**
- Pull 0 vs Pull 1 (cross-time, ~7 months)
- Pull 0 vs Pull 2 (cross-time, ~7 months)
- Pull 1 vs Pull 2 (same-day, ~2 hours)

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

DATA_DIR = 'data'
PULLS = [
    ('raw_20260417_211800', 'Pull 0 (Apr 2026)'),
    ('raw_20261105_065925', 'Pull 1 (Nov 2026a)'),
    ('raw_20261105_090331', 'Pull 2 (Nov 2026b)'),
]
PAIRS = [
    (0, 1, 'cross-time'),
    (0, 2, 'cross-time'),
    (1, 2, 'same-day'),
]
print(f'Data directory: {DATA_DIR}')
print(f'Pulls: {len(PULLS)}, Pairs: {len(PAIRS)}')

In [ ]:
def load_country_category(pull_folder, country, category):
    """Load and concatenate all chunks for a country-category pair, return monthly RSV series."""
    pattern = os.path.join(DATA_DIR, pull_folder, country, f'{category}__*__chunk*.csv')
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    dfs = []
    for f in files:
        df = pd.read_csv(f, parse_dates=['date'])
        # Average all keyword columns into a single RSV per month
        keyword_cols = [c for c in df.columns if c != 'date']
        df['rsv'] = df[keyword_cols].mean(axis=1)
        dfs.append(df[['date', 'rsv']])
    combined = pd.concat(dfs).groupby('date')['rsv'].mean().reset_index()
    return combined.sort_values('date').reset_index(drop=True)

# Discover countries and categories
sample_pull = PULLS[0][0]
countries = sorted([d for d in os.listdir(os.path.join(DATA_DIR, sample_pull))
                    if os.path.isdir(os.path.join(DATA_DIR, sample_pull, d))])
categories_set = set()
for f in os.listdir(os.path.join(DATA_DIR, sample_pull, countries[0])):
    if f.endswith('.csv'):
        categories_set.add(f.split('__')[0])
categories = sorted(categories_set)

print(f'Countries ({len(countries)}): {", ".join(countries)}')
print(f'Categories ({len(categories)}): {", ".join(categories)}')

## Compute Sampling Variability Metrics

For each country × category × pull-pair, we compute:
- **MAD**: Mean Absolute Difference of monthly RSV values
- **RMSD**: Root Mean Square Difference
- **Pearson r**: temporal pattern correlation
- **Mean difference** and **Max absolute difference**

In [ ]:
results = []

for country in countries:
    for category in categories:
        # Load all 3 pulls
        series = {}
        for i, (folder, label) in enumerate(PULLS):
            s = load_country_category(folder, country, category)
            if s is not None:
                series[i] = s
        
        # Compute pairwise metrics
        for pi, pj, interval in PAIRS:
            if pi not in series or pj not in series:
                continue
            si, sj = series[pi], series[pj]
            # Align on date
            merged = si.merge(sj, on='date', suffixes=('_a', '_b'))
            if len(merged) < 5:
                continue
            a = merged['rsv_a'].values
            b = merged['rsv_b'].values
            
            diff = a - b
            mad = np.mean(np.abs(diff))
            rmsd = np.sqrt(np.mean(diff**2))
            mean_diff = np.mean(diff)
            max_abs_diff = np.max(np.abs(diff))
            
            # Pearson r (handle constant series)
            if np.std(a) > 0 and np.std(b) > 0:
                r, p = stats.pearsonr(a, b)
            else:
                r, p = np.nan, np.nan
            
            results.append({
                'country': country,
                'category': category,
                'pull_a': pi,
                'pull_b': pj,
                'interval': interval,
                'n_months': len(merged),
                'mad': mad,
                'rmsd': rmsd,
                'pearson_r': r,
                'pearson_p': p,
                'mean_diff': mean_diff,
                'max_abs_diff': max_abs_diff,
            })

df = pd.DataFrame(results)
print(f'Total pairwise comparisons: {len(df)}')
print(f'  Cross-time: {len(df[df.interval=="cross-time"])}')
print(f'  Same-day:   {len(df[df.interval=="same-day"])}')

## Overall Summary

In [ ]:
print('=== Overall Sampling Variability ===')
print(f'N comparisons: {len(df)}')
print()
for metric in ['mad', 'rmsd', 'pearson_r', 'max_abs_diff']:
    vals = df[metric].dropna()
    label = {'mad': 'MAD (RSV points)', 'rmsd': 'RMSD (RSV points)',
             'pearson_r': 'Pearson r', 'max_abs_diff': 'Max |diff|'}[metric]
    print(f'{label}:')
    print(f'  Median = {vals.median():.3f}, Mean = {vals.mean():.3f}')
    print(f'  IQR = [{vals.quantile(0.25):.3f}, {vals.quantile(0.75):.3f}]')
    print(f'  Range = [{vals.min():.3f}, {vals.max():.3f}]')
    print()

## Summary by Interval Type

In [ ]:
print('=== By Interval Type ===')
for interval in ['same-day', 'cross-time']:
    sub = df[df.interval == interval]
    print(f'\n--- {interval} (n={len(sub)}) ---')
    for metric in ['mad', 'rmsd', 'pearson_r']:
        vals = sub[metric].dropna()
        label = {'mad': 'MAD', 'rmsd': 'RMSD', 'pearson_r': 'Pearson r'}[metric]
        print(f'  {label}: median={vals.median():.3f}, IQR=[{vals.quantile(0.25):.3f}, {vals.quantile(0.75):.3f}]')

## Summary by Country

In [ ]:
country_stats = df.groupby('country').agg(
    mad_median=('mad', 'median'),
    rmsd_median=('rmsd', 'median'),
    pearson_r_median=('pearson_r', 'median'),
    n=('mad', 'count'),
).sort_values('mad_median')

print('=== Sampling Variability by Country (sorted by median MAD) ===')
print(f'{"Country":<8} {"MAD":>8} {"RMSD":>8} {"r":>8} {"n":>4}')
print('-' * 40)
for country, row in country_stats.iterrows():
    print(f'{country:<8} {row.mad_median:8.2f} {row.rmsd_median:8.2f} {row.pearson_r_median:8.3f} {int(row.n):4d}')

## Summary by Keyword Category

In [ ]:
cat_stats = df.groupby('category').agg(
    mad_median=('mad', 'median'),
    rmsd_median=('rmsd', 'median'),
    pearson_r_median=('pearson_r', 'median'),
    n=('mad', 'count'),
).sort_values('mad_median')

print('=== Sampling Variability by Category (sorted by median MAD) ===')
print(f'{"Category":<30} {"MAD":>8} {"RMSD":>8} {"r":>8} {"n":>4}')
print('-' * 62)
for cat, row in cat_stats.iterrows():
    print(f'{cat:<30} {row.mad_median:8.2f} {row.rmsd_median:8.2f} {row.pearson_r_median:8.3f} {int(row.n):4d}')

## Visualisation: MAD by Country

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

order = country_stats.index.tolist()
positions = range(len(order))

# Box plot of MAD by country
data_by_country = [df[df.country == c]['mad'].values for c in order]
bp = ax.boxplot(data_by_country, positions=positions, widths=0.6, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7),
                medianprops=dict(color='#C44E52', linewidth=2))

ax.set_xticks(positions)
ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('MAD (RSV points)')
ax.set_title('Sampling Variability (MAD) by Country')
ax.set_xlabel('Country')
plt.tight_layout()
plt.savefig('output/fig_mad_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/fig_mad_by_country.png')

## Visualisation: Distribution of MAD and Pearson r

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {'same-day': '#4C72B0', 'cross-time': '#DD8452'}

# MAD histogram
ax = axes[0]
for interval, color in colors.items():
    sub = df[df.interval == interval]
    ax.hist(sub['mad'], bins=20, alpha=0.6, color=color, label=interval, edgecolor='white')
ax.set_xlabel('MAD (RSV points)')
ax.set_ylabel('Count')
ax.set_title('Distribution of MAD')
ax.legend()

# Pearson r histogram
ax = axes[1]
for interval, color in colors.items():
    sub = df[df.interval == interval]
    vals = sub['pearson_r'].dropna()
    ax.hist(vals, bins=20, alpha=0.6, color=color, label=interval, edgecolor='white')
ax.set_xlabel('Pearson r')
ax.set_ylabel('Count')
ax.set_title('Distribution of Pearson r')
ax.legend()

plt.tight_layout()
plt.savefig('output/fig_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/fig_distributions.png')

## Visualisation: MAD Heatmap (Country × Category)

In [ ]:
pivot = df.groupby(['country', 'category'])['mad'].median().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([c.replace('_', '\n') for c in pivot.columns], rotation=0, fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Median MAD by Country and Category')

# Add text annotations
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = 'white' if val > pivot.values.max() * 0.6 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Median MAD (RSV points)', shrink=0.8)
plt.tight_layout()
plt.savefig('output/fig_mad_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/fig_mad_heatmap.png')

## Summary and Conclusions

The results above quantify the sampling variability inherent in Google Trends RSV data.
Key findings will be summarised after execution.

In [ ]:
# Final summary
print('='*60)
print('SAMPLING VARIABILITY SUMMARY')
print('='*60)

overall_mad = df['mad'].median()
overall_rmsd = df['rmsd'].median()
overall_r = df['pearson_r'].dropna().median()

sd_mad = df[df.interval=='same-day']['mad'].median()
ct_mad = df[df.interval=='cross-time']['mad'].median()
sd_r = df[df.interval=='same-day']['pearson_r'].dropna().median()
ct_r = df[df.interval=='cross-time']['pearson_r'].dropna().median()

print(f'\nOverall (N={len(df)} comparisons):')
print(f'  Median MAD  = {overall_mad:.2f} RSV points')
print(f'  Median RMSD = {overall_rmsd:.2f} RSV points')
print(f'  Median r    = {overall_r:.3f}')

print(f'\nSame-day (≈2 h apart):')
print(f'  Median MAD  = {sd_mad:.2f} RSV points')
print(f'  Median r    = {sd_r:.3f}')

print(f'\nCross-time (≈7 months apart):')
print(f'  Median MAD  = {ct_mad:.2f} RSV points')
print(f'  Median r    = {ct_r:.3f}')

print(f'\nInterpretation:')
if overall_mad < 3:
    print(f'  Median MAD < 3 RSV points indicates minimal sampling variability.')
elif overall_mad < 5:
    print(f'  Median MAD < 5 RSV points indicates low to moderate sampling variability.')
else:
    print(f'  Median MAD ≥ 5 RSV points indicates notable sampling variability.')

if sd_mad < ct_mad:
    print(f'  Same-day variability (MAD={sd_mad:.2f}) < cross-time (MAD={ct_mad:.2f}), '
          f'as expected.')

print(f'\nConclusion: Sampling variability is {'minimal' if overall_mad < 3 else 'low' if overall_mad < 5 else 'moderate'} '
      f'(median MAD={overall_mad:.2f}), with high temporal pattern stability '
      f'(median r={overall_r:.3f}). Results support the reliability of the '
      f'Google Trends data used in the study.')